# Synthetic-to-Real Credit Card Fraud Transfer
## E1 / E2 / E3 Experiment Notebook — Kaggle GPU

This notebook implements the three core experiments from the research plan:

- **E1 — In-domain baseline**: train and test on IBM synthetic transactions (full feature set).
- **E2 — Naive direct transfer**: train on IBM using only raw columns nominally shared with ULB (`Amount`, `Time`), test on ULB with no alignment.
- **E3 — Aligned transfer**: train on IBM using a domain-invariant representation `Z` (quantile-aligned Amount + cyclical time-of-day), test on ULB mapped into the same `Z`.
- **E4 (optional)** — few-shot fine-tune of the E3 model on a small labeled ULB subset.

Each experiment reports **ROC-AUC, PR-AUC, Precision, Recall, F1, MCC, and Recall@1%FPR**, with bootstrap confidence intervals. A SHAP section compares feature importance between E1 and E3 as evidence for (or against) genuine domain-invariant signal.

**Environment assumption:** Kaggle Notebook with GPU accelerator enabled (Settings → Accelerator → GPU T4 x2 or P100). XGBoost is run with GPU histogram training; if no GPU is available, the code automatically falls back to CPU.

**Inputs expected under `/kaggle/input/`:**
- `credit_card_transactions-ibm_v2.csv` (+ optionally `sd254_cards.csv`, `sd254_users.csv`)
- `creditcard.csv` (ULB/Worldline)


## 1. Environment & Imports

In [ ]:
# ============================================================
# 1. Imports and environment
# ============================================================
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
)

import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("xgboost version:", xgb.__version__)


In [ ]:
# ============================================================
# 1b. GPU detection
# ============================================================
def detect_gpu():
    """Return XGBoost params that use GPU if available, else CPU."""
    try:
        import subprocess
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        has_gpu = result.returncode == 0
    except FileNotFoundError:
        has_gpu = False

    if has_gpu:
        print("GPU detected — using CUDA-accelerated XGBoost (tree_method='hist', device='cuda').")
        return {"tree_method": "hist", "device": "cuda"}
    else:
        print("No GPU detected — falling back to CPU ('tree_method'='hist', device='cpu').")
        return {"tree_method": "hist", "device": "cpu"}

XGB_DEVICE_PARAMS = detect_gpu()


## 2. Locate mounted datasets

In [ ]:
# ============================================================
# 2. Discover mounted CSV files (Kaggle-first, with local fallback)
# ============================================================
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]

csv_files = []
for root in SEARCH_ROOTS:
    if root.exists():
        csv_files.extend([p for p in root.rglob("*.csv") if p.is_file()])

def find_by_name(patterns, files):
    return [p for p in files if any(pat.lower() in p.name.lower() for pat in patterns)]

ulb_candidates = find_by_name(["creditcard.csv"], csv_files)
ibm_candidates = find_by_name(["credit_card_transactions-ibm_v2.csv"], csv_files)

if not ulb_candidates:
    raise FileNotFoundError("Could not find creditcard.csv (ULB/Worldline dataset).")
if not ibm_candidates:
    raise FileNotFoundError("Could not find credit_card_transactions-ibm_v2.csv (IBM dataset).")

ULB_PATH = str(ulb_candidates[0])
IBM_PATH = str(ibm_candidates[0])

print("ULB path:", ULB_PATH)
print("IBM path:", IBM_PATH)


## 3. Load and clean the IBM dataset

IBM's `Amount` field is a currency string (e.g. `"$134.09"`) and must be cleaned before use.
`Year`, `Month`, `Day`, `Time` are combined into a single `Timestamp` for time-based splitting
and behavioral feature construction. Dtypes are downcast where possible to control memory use
on a ~24M-row file.

If the file is too large for available RAM, set `IBM_SAMPLE_FRAC` below to a value < 1.0 to
subsample rows during load (documented explicitly in the results write-up as a limitation).

In [ ]:
# ============================================================
# 3. Load IBM dataset
# ============================================================
IBM_SAMPLE_FRAC = 1.0   # set < 1.0 (e.g. 0.5) if memory constrained; keep 1.0 for the full run

IBM_DTYPES = {
    "User": "int32",
    "Card": "int8",
    "Year": "int16",
    "Month": "int8",
    "Day": "int8",
    "MCC": "int32",
    "Use Chip": "category",
    "Merchant Name": "category",
    "Merchant City": "category",
    "Merchant State": "category",
    "Zip": "category",
    "Errors?": "category",
    "Is Fraud?": "category",
}

usecols = list(IBM_DTYPES.keys()) + ["Time", "Amount"]

if IBM_SAMPLE_FRAC >= 1.0:
    ibm_df = pd.read_csv(IBM_PATH, usecols=usecols, dtype=IBM_DTYPES, low_memory=False)
else:
    # Reservoir-style subsample via chunked reads to control memory
    chunks = []
    for chunk in pd.read_csv(IBM_PATH, usecols=usecols, dtype=IBM_DTYPES,
                              chunksize=1_000_000, low_memory=False):
        chunks.append(chunk.sample(frac=IBM_SAMPLE_FRAC, random_state=RANDOM_SEED))
    ibm_df = pd.concat(chunks, ignore_index=True)

print("IBM raw shape:", ibm_df.shape)

# Clean Amount: strip "$" and thousands separators
ibm_df["Amount"] = pd.to_numeric(
    ibm_df["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)

# Build Timestamp from Year/Month/Day/Time ("HH:MM")
ibm_df["Timestamp"] = pd.to_datetime(
    ibm_df["Year"].astype(str) + "-" +
    ibm_df["Month"].astype(str).str.zfill(2) + "-" +
    ibm_df["Day"].astype(str).str.zfill(2) + " " +
    ibm_df["Time"].astype(str),
    errors="coerce",
)

# Binary fraud target
ibm_df["Fraud"] = (
    ibm_df["Is Fraud?"].astype(str).str.strip().str.lower().isin(["yes", "true", "1", "fraud", "y"])
).astype(int)

# Drop rows where cleaning failed
before = len(ibm_df)
ibm_df = ibm_df.dropna(subset=["Amount", "Timestamp"]).reset_index(drop=True)
print(f"Dropped {before - len(ibm_df):,} rows with invalid Amount/Timestamp ({(before-len(ibm_df))/before*100:.4f}%)")

print("\nIBM fraud rate: {:.4f}%".format(ibm_df["Fraud"].mean() * 100))
ibm_df.head()


## 4. IBM behavioral feature engineering (full feature set — used only in E1)

These features use IBM's entity history (`User`, `Card`, `Merchant Name`) and therefore
**cannot** be replicated on ULB, which has no entity identifiers. They are used only for the
in-domain E1 baseline, not for the cross-domain experiments.

In [ ]:
# ============================================================
# 4. Full IBM feature engineering (in-domain only)
# ============================================================
ibm_df = ibm_df.sort_values(["User", "Timestamp"], kind="mergesort").reset_index(drop=True)

# Inter-transaction time gap per user
ibm_df["Delta_t_seconds"] = (
    ibm_df.groupby("User")["Timestamp"].diff().dt.total_seconds()
)
ibm_df.loc[ibm_df["Delta_t_seconds"] < 0, "Delta_t_seconds"] = np.nan

# Previous amount / amount change per user
ibm_df["Previous_amount"] = ibm_df.groupby("User")["Amount"].shift(1)
ibm_df["Amount_change"] = ibm_df["Amount"] - ibm_df["Previous_amount"]

# Rolling per-user amount deviation (expanding mean/std, shifted to avoid leakage)
grp = ibm_df.groupby("User")["Amount"]
roll_mean = grp.apply(lambda s: s.shift(1).expanding().mean()).reset_index(level=0, drop=True)
roll_std = grp.apply(lambda s: s.shift(1).expanding().std()).reset_index(level=0, drop=True)
ibm_df["Amount_deviation"] = (ibm_df["Amount"] - roll_mean) / roll_std.replace(0, np.nan)

# Merchant novelty: has this user transacted with this merchant before?
ibm_df["Merchant_seen_before"] = (
    ibm_df.groupby(["User", "Merchant Name"]).cumcount() > 0
).astype(int)

# Calendar/time features
ibm_df["Hour"] = ibm_df["Timestamp"].dt.hour
ibm_df["DayOfWeek"] = ibm_df["Timestamp"].dt.dayofweek

print("IBM feature engineering complete. Columns:")
print(ibm_df.columns.tolist())
ibm_df[["Amount", "Delta_t_seconds", "Amount_change", "Amount_deviation",
        "Merchant_seen_before", "Hour", "DayOfWeek", "Fraud"]].describe()


## 5. Load ULB / Worldline dataset

`V1`–`V28` are PCA-anonymized and are **not** assigned semantic meaning here or used in the
cross-domain representation, per the plan. `Time` is elapsed seconds from an arbitrary start
over ~2 days; `Time % 86400` is used only as a coarse proxy for time-of-day, which is a
documented assumption, not a verified fact.

In [ ]:
# ============================================================
# 5. Load ULB dataset
# ============================================================
ulb_df = pd.read_csv(ULB_PATH, low_memory=False)
ulb_df = ulb_df.rename(columns={"Class": "Fraud"})

print("ULB shape:", ulb_df.shape)
print("ULB fraud rate: {:.4f}%".format(ulb_df["Fraud"].mean() * 100))

# Time-of-day proxy (documented assumption)
ulb_df["SecondsIntoDay"] = ulb_df["Time"] % 86400

ulb_df.head()


## 6. Build the shared representation `Z` (domain alignment)

The alignment step:

1. Fit a `QuantileTransformer` on **IBM training-split Amount only**, mapping it to a
   uniform [0, 1] scale. Apply the same fitted transformer to both IBM and ULB amounts.
   This is the "lightweight domain alignment" — it puts both currencies'/datasets' amount
   distributions on a comparable relative scale without assuming a shared absolute currency
   meaning.
2. Convert time to cyclical sine/cosine components so both datasets contribute a comparable
   "time-of-day" signal regardless of their different absolute time semantics.

`Z = [Amount_q, Hour_sin, Hour_cos]` is intentionally small — see the research plan's
discussion of why velocity/behavioral-deviation features cannot be part of a defensible
shared representation (ULB has no entity IDs).

In [ ]:
# ============================================================
# 6. Domain-aligned shared representation Z
# ============================================================

def add_cyclical_hour(df, hour_col):
    hours = df[hour_col].astype(float)
    df["Hour_sin"] = np.sin(2 * np.pi * hours / 24.0)
    df["Hour_cos"] = np.cos(2 * np.pi * hours / 24.0)
    return df

ibm_df = add_cyclical_hour(ibm_df, "Hour")

ulb_df["Hour_of_day_proxy"] = ulb_df["SecondsIntoDay"] / 3600.0
ulb_df = add_cyclical_hour(ulb_df, "Hour_of_day_proxy")

# Time-based split for IBM (fit the amount transformer on TRAIN only to avoid leakage)
ibm_df = ibm_df.sort_values("Timestamp").reset_index(drop=True)
split_idx = int(len(ibm_df) * 0.8)
ibm_train_idx = ibm_df.index[:split_idx]
ibm_test_idx = ibm_df.index[split_idx:]

amount_qt = QuantileTransformer(output_distribution="uniform", random_state=RANDOM_SEED)
amount_qt.fit(ibm_df.loc[ibm_train_idx, ["Amount"]])

ibm_df["Amount_q"] = amount_qt.transform(ibm_df[["Amount"]])
ulb_df["Amount_q"] = amount_qt.transform(ulb_df[["Amount"]].clip(lower=0))

Z_COLS = ["Amount_q", "Hour_sin", "Hour_cos"]

print("Z columns:", Z_COLS)
print("\nIBM Z sample:")
print(ibm_df[Z_COLS + ["Fraud"]].head())
print("\nULB Z sample:")
print(ulb_df[Z_COLS + ["Fraud"]].head())


## 7. Train/test splits

- **IBM**: time-based 80/20 split (train on earlier transactions, test on later ones) —
  used for E1 and as the training set for E2/E3.
- **ULB**: used in full, unbalanced, as an external test set for E2/E3 (never used for
  training in E2/E3, except for the small labeled subset carved out for optional E4).

In [ ]:
# ============================================================
# 7. Build split frames
# ============================================================
IBM_FULL_FEATURES = [
    "Amount", "Delta_t_seconds", "Amount_change", "Amount_deviation",
    "Merchant_seen_before", "Hour", "DayOfWeek",
]

ibm_train = ibm_df.loc[ibm_train_idx].copy()
ibm_test = ibm_df.loc[ibm_test_idx].copy()

print("IBM train shape:", ibm_train.shape, "| fraud rate: {:.4f}%".format(ibm_train["Fraud"].mean()*100))
print("IBM test shape :", ibm_test.shape, "| fraud rate: {:.4f}%".format(ibm_test["Fraud"].mean()*100))
print("ULB (external) shape:", ulb_df.shape, "| fraud rate: {:.4f}%".format(ulb_df["Fraud"].mean()*100))


## 8. Evaluation utilities

Computes ROC-AUC, PR-AUC, Precision, Recall, F1, MCC, and Recall@1%FPR, plus bootstrap
confidence intervals. Accuracy is deliberately excluded from the reported metrics (with
severe class imbalance it is not informative — a constant "not fraud" prediction scores
~99%+ while catching zero fraud).

In [ ]:
# ============================================================
# 8. Metrics
# ============================================================

def recall_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    idx = np.searchsorted(fpr, target_fpr, side="right") - 1
    idx = max(idx, 0)
    return tpr[idx], thresholds[idx]

def best_f1_threshold(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1]) if len(thresholds) > 0 else 0
    return thresholds[best_idx] if len(thresholds) > 0 else 0.5

def compute_metrics(y_true, y_score, label=""):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    roc_auc = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)

    thr_f1 = best_f1_threshold(y_true, y_score)
    y_pred_f1 = (y_score >= thr_f1).astype(int)

    recall_1fpr, thr_1fpr = recall_at_fpr(y_true, y_score, target_fpr=0.01)
    y_pred_1fpr = (y_score >= thr_1fpr).astype(int)

    metrics = {
        "experiment": label,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision_at_bestF1": precision_score(y_true, y_pred_f1, zero_division=0),
        "recall_at_bestF1": recall_score(y_true, y_pred_f1, zero_division=0),
        "f1_at_bestF1": f1_score(y_true, y_pred_f1, zero_division=0),
        "mcc_at_bestF1": matthews_corrcoef(y_true, y_pred_f1),
        "recall_at_1pct_fpr": recall_1fpr,
        "precision_at_1pct_fpr": precision_score(y_true, y_pred_1fpr, zero_division=0),
    }
    return metrics

def bootstrap_ci(y_true, y_score, metric_fn, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, ys = y_true[idx], y_score[idx]
        if yt.sum() == 0 or yt.sum() == n:
            continue
        vals.append(metric_fn(yt, ys))
    vals = np.array(vals)
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

all_results = []


## 9. E1 — In-domain baseline (IBM → IBM)

Trains XGBoost on IBM's full engineered feature set, tests on the held-out (later-in-time)
IBM split. This establishes the achievable ceiling with all available signal.

In [ ]:
# ============================================================
# 9. E1 — In-domain IBM baseline
# ============================================================
X_train_e1 = ibm_train[IBM_FULL_FEATURES].fillna(-999)
y_train_e1 = ibm_train["Fraud"]
X_test_e1 = ibm_test[IBM_FULL_FEATURES].fillna(-999)
y_test_e1 = ibm_test["Fraud"]

scale_pos_weight_e1 = (y_train_e1 == 0).sum() / max((y_train_e1 == 1).sum(), 1)

model_e1 = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_e1,
    eval_metric="aucpr",
    random_state=RANDOM_SEED,
    **XGB_DEVICE_PARAMS,
)
model_e1.fit(X_train_e1, y_train_e1)

score_e1 = model_e1.predict_proba(X_test_e1)[:, 1]
metrics_e1 = compute_metrics(y_test_e1, score_e1, label="E1: IBM -> IBM (in-domain)")
ci_pr_e1 = bootstrap_ci(y_test_e1, score_e1, average_precision_score)
metrics_e1["pr_auc_95ci"] = ci_pr_e1
all_results.append(metrics_e1)

print(json.dumps({k: (v if not isinstance(v, tuple) else list(v)) for k, v in metrics_e1.items()}, indent=2, default=str))


## 10. E2 — Naive direct transfer (IBM → ULB, no alignment)

Trains on IBM using only the raw columns nominally shared with ULB (`Amount`, `Time`, taken
at face value with no rescaling or semantic reconciliation), then tests directly on ULB.
This is the naive baseline / transfer floor.

In [ ]:
# ============================================================
# 10. E2 — Naive direct transfer
# ============================================================
NAIVE_COLS = ["Amount", "Time"]

# IBM 'Time' here is reinterpreted naively as seconds-into-day for a face-value match to ULB
ibm_df["Time_seconds_naive"] = ibm_df["Hour"] * 3600  # coarse, deliberately naive
ibm_train["Time_seconds_naive"] = ibm_df.loc[ibm_train.index, "Time_seconds_naive"]
ibm_test["Time_seconds_naive"] = ibm_df.loc[ibm_test.index, "Time_seconds_naive"]

X_train_e2 = ibm_train[["Amount", "Time_seconds_naive"]].rename(columns={"Time_seconds_naive": "Time"})
y_train_e2 = ibm_train["Fraud"]

X_test_e2 = ulb_df[["Amount", "SecondsIntoDay"]].rename(columns={"SecondsIntoDay": "Time"})
y_test_e2 = ulb_df["Fraud"]

scale_pos_weight_e2 = (y_train_e2 == 0).sum() / max((y_train_e2 == 1).sum(), 1)

model_e2 = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_e2,
    eval_metric="aucpr",
    random_state=RANDOM_SEED,
    **XGB_DEVICE_PARAMS,
)
model_e2.fit(X_train_e2, y_train_e2)

score_e2 = model_e2.predict_proba(X_test_e2)[:, 1]
metrics_e2 = compute_metrics(y_test_e2, score_e2, label="E2: IBM -> ULB (naive transfer)")
ci_pr_e2 = bootstrap_ci(y_test_e2, score_e2, average_precision_score)
metrics_e2["pr_auc_95ci"] = ci_pr_e2
all_results.append(metrics_e2)

print(json.dumps({k: (v if not isinstance(v, tuple) else list(v)) for k, v in metrics_e2.items()}, indent=2, default=str))


## 11. E3 — Aligned transfer (IBM → Z → ULB)

Trains XGBoost on IBM mapped into the shared representation `Z` (quantile-aligned Amount +
cyclical time-of-day), tests on ULB mapped into the same `Z`. This is the paper's core
result — the comparison that matters is **E3 vs E2**, not E3's absolute score.

In [ ]:
# ============================================================
# 11. E3 — Aligned transfer via Z
# ============================================================
X_train_e3 = ibm_train[Z_COLS]
y_train_e3 = ibm_train["Fraud"]

X_test_e3 = ulb_df[Z_COLS]
y_test_e3 = ulb_df["Fraud"]

scale_pos_weight_e3 = (y_train_e3 == 0).sum() / max((y_train_e3 == 1).sum(), 1)

model_e3 = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_e3,
    eval_metric="aucpr",
    random_state=RANDOM_SEED,
    **XGB_DEVICE_PARAMS,
)
model_e3.fit(X_train_e3, y_train_e3)

score_e3 = model_e3.predict_proba(X_test_e3)[:, 1]
metrics_e3 = compute_metrics(y_test_e3, score_e3, label="E3: IBM -> Z -> ULB (aligned transfer)")
ci_pr_e3 = bootstrap_ci(y_test_e3, score_e3, average_precision_score)
metrics_e3["pr_auc_95ci"] = ci_pr_e3
all_results.append(metrics_e3)

print(json.dumps({k: (v if not isinstance(v, tuple) else list(v)) for k, v in metrics_e3.items()}, indent=2, default=str))

# Significance check: E3 vs E2 (paired bootstrap on PR-AUC delta), only valid since both
# are scored on the same ULB test set
def paired_bootstrap_delta(y_true, score_a, score_b, metric_fn, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    n = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt = y_true[idx]
        if yt.sum() == 0 or yt.sum() == n:
            continue
        deltas.append(metric_fn(yt, score_a[idx]) - metric_fn(yt, score_b[idx]))
    deltas = np.array(deltas)
    return deltas.mean(), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)

delta_mean, delta_lo, delta_hi = paired_bootstrap_delta(
    y_test_e3, score_e3, score_e2, average_precision_score
)
print(f"\nE3 - E2 PR-AUC delta: {delta_mean:.4f}  (95% CI: [{delta_lo:.4f}, {delta_hi:.4f}])")
print("If this interval excludes 0, alignment provides a statistically significant improvement over naive transfer.")


## 12. E4 (optional) — Few-shot fine-tuning on a small ULB labeled subset

Carves out a small stratified subset of ULB (e.g. 10% of ULB, preserving fraud rate) to
continue training the E3 model, then evaluates on the remaining 90% of ULB. Skip this cell
if the paper's scope stays at E1–E3.

In [ ]:
# ============================================================
# 12. E4 — Few-shot fine-tune (optional)
# ============================================================
RUN_E4 = True
FEWSHOT_FRAC = 0.10

if RUN_E4:
    ulb_fewshot, ulb_holdout = train_test_split(
        ulb_df, test_size=(1 - FEWSHOT_FRAC), stratify=ulb_df["Fraud"], random_state=RANDOM_SEED
    )

    X_fewshot = ulb_fewshot[Z_COLS]
    y_fewshot = ulb_fewshot["Fraud"]
    X_holdout = ulb_holdout[Z_COLS]
    y_holdout = ulb_holdout["Fraud"]

    # Continue training from E3's learned trees on the small real-labeled subset
    model_e4 = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.03,
        scale_pos_weight=(y_fewshot == 0).sum() / max((y_fewshot == 1).sum(), 1),
        eval_metric="aucpr",
        random_state=RANDOM_SEED,
        **XGB_DEVICE_PARAMS,
    )
    model_e4.fit(X_fewshot, y_fewshot, xgb_model=model_e3.get_booster())

    score_e4 = model_e4.predict_proba(X_holdout)[:, 1]
    metrics_e4 = compute_metrics(y_holdout, score_e4, label="E4: IBM -> Z -> ULB (few-shot fine-tuned)")
    ci_pr_e4 = bootstrap_ci(y_holdout, score_e4, average_precision_score)
    metrics_e4["pr_auc_95ci"] = ci_pr_e4
    all_results.append(metrics_e4)

    print(json.dumps({k: (v if not isinstance(v, tuple) else list(v)) for k, v in metrics_e4.items()}, indent=2, default=str))
else:
    print("E4 skipped (RUN_E4 = False).")


## 13. Results summary table

In [ ]:
# ============================================================
# 13. Compile results
# ============================================================
results_df = pd.DataFrame(all_results)
cols_order = [
    "experiment", "roc_auc", "pr_auc", "pr_auc_95ci",
    "precision_at_bestF1", "recall_at_bestF1", "f1_at_bestF1", "mcc_at_bestF1",
    "recall_at_1pct_fpr", "precision_at_1pct_fpr",
]
results_df = results_df[[c for c in cols_order if c in results_df.columns]]
results_df


## 14. Explainability (SHAP)

Compares feature importance between **E1 (in-domain)** and **E3 (aligned transfer)** on
their respective shared `Z` features. Note: E1's model was trained on the full IBM feature
set — for a fair like-for-like SHAP comparison on shared features, a second E1-style model
restricted to `Z_COLS` is trained here purely for this diagnostic (`model_e1_zonly`).

In [ ]:
# ============================================================
# 14. SHAP analysis
# ============================================================
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "shap", "--quiet"])
    import shap

# Train a Z-only IBM model for a fair comparison against E3 on the same feature set
model_e1_zonly = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_e1,
    eval_metric="aucpr", random_state=RANDOM_SEED, **XGB_DEVICE_PARAMS,
)
model_e1_zonly.fit(ibm_train[Z_COLS], ibm_train["Fraud"])

explainer_e1 = shap.TreeExplainer(model_e1_zonly)
shap_values_e1 = explainer_e1.shap_values(ibm_test[Z_COLS].sample(min(20000, len(ibm_test)), random_state=RANDOM_SEED))

explainer_e3 = shap.TreeExplainer(model_e3)
shap_values_e3 = explainer_e3.shap_values(ulb_df[Z_COLS].sample(min(20000, len(ulb_df)), random_state=RANDOM_SEED))

print("SHAP values computed for E1 (Z-only, in-domain) and E3 (aligned transfer).")


In [ ]:
# Global importance — E1 (Z-only, in-domain)
shap.summary_plot(
    shap_values_e1,
    ibm_test[Z_COLS].sample(min(20000, len(ibm_test)), random_state=RANDOM_SEED),
    plot_type="bar",
    show=True,
)


In [ ]:
# Global importance — E3 (aligned transfer)
shap.summary_plot(
    shap_values_e3,
    ulb_df[Z_COLS].sample(min(20000, len(ulb_df)), random_state=RANDOM_SEED),
    plot_type="bar",
    show=True,
)


In [ ]:
# Dependence plots for each shared Z feature — E1 vs E3 side by side (direction consistency check)
for feat in Z_COLS:
    print(f"--- {feat} ---")
    shap.dependence_plot(
        feat, shap_values_e1,
        ibm_test[Z_COLS].sample(min(20000, len(ibm_test)), random_state=RANDOM_SEED),
        show=True,
    )
    shap.dependence_plot(
        feat, shap_values_e3,
        ulb_df[Z_COLS].sample(min(20000, len(ulb_df)), random_state=RANDOM_SEED),
        show=True,
    )


## 15. Save artifacts

In [ ]:
# ============================================================
# 15. Save models and results
# ============================================================
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

results_df.to_csv(OUT_DIR / "syn2real_results_summary.csv", index=False)

model_e1.save_model(str(OUT_DIR / "model_e1_indomain.json"))
model_e2.save_model(str(OUT_DIR / "model_e2_naive_transfer.json"))
model_e3.save_model(str(OUT_DIR / "model_e3_aligned_transfer.json"))
if RUN_E4:
    model_e4.save_model(str(OUT_DIR / "model_e4_fewshot.json"))

print("Saved results and models to:", OUT_DIR.resolve())
